<a href="https://colab.research.google.com/github/TVTruong777/DL261-Assignment/blob/main/assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Assignment 1')

In [ ]:
import torch
import pymodels
import utils
import importlib

importlib.reload(utils)
importlib.reload(pymodels)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
using device cuda


In [ ]:
from torch.utils.data import random_split, DataLoader
import torch.nn as nn
import torchvision
import torch

transform = torchvision.transforms.ToTensor()
train_set = torchvision.datasets.FashionMNIST(root='./data', transform = transform, download = True, train = True)
test_set = torchvision.datasets.FashionMNIST(root = './data', transform = transform, download = True, train = False)

m_test = len(test_set)
m_val = m_test
m_train = len(train_set) - m_val
total_samples = m_train + m_val + m_test
best_models = {}

generator_67 = torch.Generator().manual_seed(67)
xy_train, xy_val = random_split(train_set, [m_train, m_val], generator_67)

print(f"Number of samples in training set: {len(xy_train)}")
print(f"Number of samples in validation set: {len(xy_val)}")
print(f"Number of samples in test: {len(test_set)}")
print(f"Propotion: {m_train * 100 / total_samples:.2f}/{m_val * 100 / total_samples:.2f}/{m_test * 100 / total_samples:.2f}")

In [ ]:
from pymodels import LinearModel, MLP, ConvNet, LSTMnet, VisionTransformer
from utils import fit_compile
import torch.nn as nn

train_loader = DataLoader(dataset = xy_train, batch_size = 64, shuffle = True, generator = generator_67)
val_loader = DataLoader(dataset = xy_val, batch_size = 64, shuffle = False)
epochs = 50
max_patience = 5

In [ ]:
lin_model = LinearModel()
lin_model = lin_model.to(device)
lin_path = 'best_linear_model.pth'
SparseCategoricalCrossentropy = nn.CrossEntropyLoss()
adam_optimizer = torch.optim.Adam(lin_model.parameters(), lr = 0.001)

lin_model, train_track, validation_track, lin_train_time = fit_compile(model = lin_model,
                                                                epochs = epochs,
                                                                max_patience = max_patience,
                                                                optimizer = adam_optimizer,
                                                                loss_fn = SparseCategoricalCrossentropy,
                                                                save_path = lin_path,
                                                                train_loader = train_loader,
                                                                val_loader = val_loader)
best_models["Linear Model"] = (lin_model, train_track, validation_track, lin_train_time)

In [ ]:
mlp_model = MLP()
mlp_model = mlp_model.to(device)
mlp_path = 'best_mlp_model.pth'
SparseCategoricalCrossentropy = nn.CrossEntropyLoss()
mlp_adam_optimizer = torch.optim.Adam(mlp_model.parameters(), lr = 0.001)

best_mlp_model, mlp_train_track, mlp_val_track, mlp_train_time = fit_compile(model = mlp_model,
                                                            loss_fn = SparseCategoricalCrossentropy,
                                                            optimizer = mlp_adam_optimizer,
                                                            epochs = epochs,
                                                            max_patience = max_patience,
                                                            train_loader = train_loader,
                                                            val_loader = val_loader,
                                                            save_path = mlp_path)
best_models["MLP Model"] = (best_mlp_model, mlp_train_track, mlp_val_track,  mlp_train_time)

In [ ]:
conv_model = ConvNet()
conv_model = conv_model.to(device)
conv_path = 'best_conv_model.pth'
SparseCategoricalCrossentropy = nn.CrossEntropyLoss()
conv_adam_optimizer = torch.optim.Adam(conv_model.parameters(), lr = 0.001)

best_conv_model, conv_train_track, conv_val_track, conv_train_time = fit_compile(model = conv_model,
                                                            loss_fn = SparseCategoricalCrossentropy,
                                                            optimizer = conv_adam_optimizer,
                                                            epochs = epochs,
                                                            max_patience = max_patience,
                                                            train_loader = train_loader,
                                                            val_loader = val_loader,
                                                            save_path = conv_path)
best_models["Conv Model"] = (best_conv_model, conv_train_track, conv_val_track,  conv_train_time)

In [ ]:
lstm_model = LSTMnet()
lstm_path = "best_lstm_path"
lstm_model = lstm_model.to(device)
SparseCategoricalCrossentropy = nn.CrossEntropyLoss()
lstm_adam_optimizer = torch.optim.Adam(lstm_model.parameters(), lr = 0.001)
best_lstm, lstm_train_track, lstm_val_track, lstm_train_time = fit_compile(model = lstm_model,
                                                                           loss_fn = SparseCategoricalCrossentropy,
                                                                           optimizer = lstm_adam_optimizer,
                                                                           epochs = epochs,
                                                                           max_patience = max_patience,
                                                                           train_loader= train_loader,
                                                                           val_loader = val_loader,
                                                                           save_path = lstm_path)
best_models["LSTM Model"] = (best_lstm, lstm_train_track, lstm_val_track, lstm_train_time)

In [ ]:
vision_transformer = VisionTransformer()
vision_transformer = vision_transformer.to(device)
vt_path = "best_vt_path"
SparseCategoricalCrossEntropy = nn.CrossEntropyLoss()
vt_adam_optimizer = torch.optim.AdamW(vision_transformer.parameters(), lr = 0.0001, weight_decay = 0.01)
vt_epochs = 300
vt_max_patience = 10
best_vt, vt_train_track, vt_val_track, vt_train_time = fit_compile(model = vision_transformer,
                                                                   loss_fn = SparseCategoricalCrossEntropy,
                                                                   optimizer = vt_adam_optimizer,
                                                                   epochs = vt_epochs,
                                                                   max_patience = vt_max_patience,
                                                                   train_loader = train_loader,
                                                                   val_loader = val_loader,
                                                                   save_path = vt_path)
best_models["Vision Transformer"] = (best_vt, vt_train_track, vt_val_track, vt_train_time)

In [ ]:
from utils import evaluate_model_metrics

model_result = evaluate_model_metrics(best_models = best_models, loader = val_loader)

In [ ]:
from utils import plot_models_curve

plot_models_curve(best_models)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame.from_dict(model_result, orient = 'index', columns = ['Accuracy', 'Macro F1', 'Weighted F1', 'Train Time (s)', 'Inference Time (s)', 'Total Parameters'])

fig, axes = plt.subplots(nrows = 1, ncols = 4, figsize = (18, 5))

df[['Accuracy', 'Macro F1', 'Weighted F1']].plot(kind = 'bar', ax = axes[0] ,colormap = 'viridis')
axes[0].set_title('Classification Scores')
axes[0].set_ylim(0.7, 1.0)
axes[0].tick_params(axis='x', rotation = 0)

df[['Train Time (s)']].plot(kind = 'bar', ax = axes[1], color = 'coral', legend = False)
axes[1].set_title('Training Time (Lower is Better)')
axes[1].set_ylabel('Seconds')
axes[1].tick_params(axis='x', rotation=0)

df[['Inference Time (s)']].plot(kind = 'bar', ax = axes[2], color = 'skyblue', legend = False)
axes[2].set_title('Inference Time (Lower is Better)')
axes[2].set_ylabel('Seconds')
axes[2].tick_params(axis='x', rotation=0)

df[['Total Parameters']].plot(kind = 'bar', ax = axes[3], color = 'grey', legend = False)
axes[3].set_title('Total Parameters')
axes[3].set_ylabel('Number of Parameters')
axes[3].set_yscale('log')
axes[3].tick_params(axis = 'x', rotation = 0)

plt.tight_layout()
plt.show()

In [ ]:
from typing import Literal
import math

def error_analysis(model_name: Literal["Conv Model", "LSTM Model", "Linear Model", "MLP Model", "Vision Transformer"], loader):
  model = best_models[model_name][0]
  model.eval()
  images, labels = next(iter(loader))
  images = images.to(device)
  labels = labels.to(device)

  with torch.no_grad():
    logits = model(images)
    pred = logits.argmax(dim = 1)
    incorrect_indices = torch.where(pred != labels)[0]

  class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress',
                 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

  display_error(images = images,
                classes = class_names,
                indices = incorrect_indices,
                predictions = pred,
                labels = labels,
                num_imgs = len(incorrect_indices))

def display_error(images, classes, indices, predictions, labels, num_imgs):
  fig, axes = plt.subplots(nrows = math.ceil(num_imgs / 4), ncols = 4, figsize = (15, 5))

  for i, ax in enumerate(axes.flatten()):
    if i < len(indices):
      idx = indices[i]
      img = images[idx].cpu().squeeze().numpy()
      predicted_class = classes[predictions[idx].item()]
      actual_class = classes[labels[idx].item()]
      ax.imshow(img, cmap = 'grey')
      ax.set_title(f"Label:{actual_class}; Predict:{predicted_class}", color = 'green' if predicted_class == actual_class else 'red', fontsize = 11)
      ax.axis('off')
    else:
      ax.axis('off')
  plt.tight_layout()
  plt.show()

In [ ]:
error_analysis("Conv Model", val_loader)